# Ejercicio: Controlando tu Agente

Este ejercicio tiene dos partes:

- **Parte 1:** Modificar el agentic loop para agregar control de presupuesto
- **Parte 2:** Modificar el system prompt para cambiar el comportamiento del agente

**Reglas:**
- Solo puedes modificar las secciones marcadas con `# TODO`
- No cambies nada fuera de esas secciones
- Ejecuta las celdas de validación al final de cada parte para verificar tu solución

---
## Setup (no modificar)

In [1]:
import os
import json
import httpx
import urllib.parse
from openai import OpenAI
from dotenv import load_dotenv
from ddgs import DDGS

load_dotenv()

BACKEND = "nvidia"

CONFIGS = {
    "ollama":    {"base_url": "http://localhost:11434/v1",         "api_key": "ollama",                          "model": "qwen2.5:7b"},
    "anthropic": {"base_url": "https://api.anthropic.com/v1",     "api_key": os.getenv("ANTHROPIC_API_KEY",""), "model": "claude-3-5-haiku-20241022"},
    "openai":    {"base_url": None,                                "api_key": os.getenv("OPENAI_API_KEY",""),   "model": "gpt-4o-mini"},
    "groq":      {"base_url": "https://api.groq.com/openai/v1",   "api_key": os.getenv("GROQ_API_KEY",""),     "model": "llama-3.3-70b-versatile"},
    "nvidia":    {"base_url": "https://integrate.api.nvidia.com/v1","api_key": os.getenv("NVIDIA_API_KEY",""), "model": "meta/llama-3.3-70b-instruct"},
}

cfg = CONFIGS[BACKEND]
client_kwargs = {"api_key": cfg["api_key"]}
if cfg["base_url"]:
    client_kwargs["base_url"] = cfg["base_url"]

client = OpenAI(**client_kwargs)
MODEL  = cfg["model"]
print(f"✅ Backend: {BACKEND.upper()} | Modelo: {MODEL}")

✅ Backend: NVIDIA | Modelo: meta/llama-3.3-70b-instruct


In [2]:
# Herramientas (no modificar)

def web_search(query: str) -> str:
    """Busca información actualizada en la web."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No se encontraron resultados."
        return "\n\n".join(f"**{r['title']}**\n{r['body']}" for r in results)
    except Exception as e:
        return f"Error al buscar: {e}"


def get_weather(city: str) -> str:
    """Obtiene el clima actual de una ciudad usando wttr.in."""
    try:
        r = httpx.get(f"https://wttr.in/{urllib.parse.quote(city)}?format=3", timeout=8)
        return r.text.strip()
    except Exception as e:
        return f"Error: {e}"


def calculate(expression: str) -> str:
    """Evalúa una expresión matemática simple."""
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return "Error: solo expresiones numéricas básicas."
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


TOOL_REGISTRY = {
    "web_search": web_search,
    "get_weather": get_weather,
    "calculate":  calculate,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": (
                "Busca información actualizada en la web. "
                "Úsala cuando necesites datos recientes o hechos que podrían "
                "haber cambiado después de tu fecha de corte."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "La consulta de búsqueda."}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Obtiene el clima actual de una ciudad específica.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Nombre de la ciudad."}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evalúa expresiones matemáticas. Úsala para cálculos precisos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Expresión matemática."}
                },
                "required": ["expression"]
            }
        }
    }
]

print(f"✅ {len(TOOLS)} herramientas: {[t['function']['name'] for t in TOOLS]}")

✅ 3 herramientas: ['web_search', 'get_weather', 'calculate']


---
## 🔄 Agentic Loop base (no modificar)

Esta es la versión original de `run_agent` que ya conoces.
La usarás como referencia para la Parte 1.

In [3]:
def run_agent(
    user_message: str,
    system: str = None,
    tools: list = None,
    tool_registry: dict = None,
    verbose: bool = True,
    label: str = "Agente",
) -> str:
    tools         = tools         if tools         is not None else TOOLS
    tool_registry = tool_registry if tool_registry is not None else TOOL_REGISTRY

    if system is None:
        system = (
            "Eres un asistente útil con acceso a herramientas. "
            "Usa las herramientas SOLO cuando necesites información que cambia con el tiempo. "
            "Para conocimiento general responde directamente. "
            "Responde en el idioma del usuario."
        )

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_message},
    ]

    if verbose:
        print(f"\n{'─'*60}")
        print(f"  [{label}] 👤 {user_message[:120]}")

    for iteration in range(1, 11):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools if tools else None,
            tool_choice="auto" if tools else None,
        )
        choice        = response.choices[0]
        finish_reason = choice.finish_reason

        if verbose:
            print(f"  [{label}] iter={iteration} finish_reason={finish_reason}")

        if finish_reason == "stop":
            text = choice.message.content or ""
            if verbose:
                preview = text[:200] + "..." if len(text) > 200 else text
                print(f"  [{label}] 🤖 {preview}")
            return text

        if finish_reason == "tool_calls":
            assistant_msg = choice.message
            messages.append(assistant_msg)
            for tc in assistant_msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)
                if verbose:
                    print(f"  [{label}] 🔧 {tool_name}({json.dumps(tool_args, ensure_ascii=False)})")
                fn     = tool_registry.get(tool_name)
                result = fn(**tool_args) if fn else f"Error: herramienta '{tool_name}' no encontrada."
                if verbose:
                    preview = result[:150] + "..." if len(result) > 150 else result
                    print(f"  [{label}]    → {preview}")
                messages.append({
                    "role":         "tool",
                    "tool_call_id": tc.id,
                    "content":      result,
                })
        else:
            break

    return "[Agente detenido: límite de iteraciones]"


print("✅ run_agent base definido.")

✅ run_agent base definido.


---
## PARTE 1: Presupuesto de Tool Calls

### Contexto

En producción, cada tool call tiene un costo: latencia, dinero (tokens), o
riesgo (una tool que escribe en una base de datos). Necesitamos poder limitar
cuántas veces el agente puede llamar herramientas en una sola ejecución.

### Tu tarea

Implementa `run_agent_with_budget`, una versión modificada de `run_agent` que:

1. Acepta un parámetro `max_tool_calls: int`
2. Lleva un contador de cuántas tool calls se han ejecutado **en total**
   (no iteraciones — en una sola iteración pueden haber múltiples tool calls)
3. Cuando el contador alcanza `max_tool_calls`, **detiene el loop inmediatamente**
   y retorna la respuesta parcial que tenga hasta ese momento, con un aviso
4. Imprime el presupuesto restante en cada tool call cuando `verbose=True`

### Pistas

- El contador debe incrementarse **dentro** del for que recorre `assistant_msg.tool_calls`
- La condición de corte va **antes** de ejecutar la función Python, no después
- Cuando se corta por presupuesto, el loop no puede continuar normalmente
  porque el historial quedó en estado inconsistente (hay tool_calls sin tool_result).
  Retorna un string descriptivo en ese caso.

In [ ]:
def run_agent_with_budget(
    user_message: str,
    max_tool_calls: int = 3,
    system: str = None,
    tools: list = None,
    tool_registry: dict = None,
    verbose: bool = True,
    label: str = "Agente",
) -> str:
    """
    Versión de run_agent con presupuesto máximo de tool calls.

    Args:
        user_message:   Pregunta o instrucción del usuario.
        max_tool_calls: Número máximo de tool calls permitidas en total.
        system:         System prompt opcional.
        tools:          Lista de tool schemas. Por defecto: TOOLS.
        tool_registry:  Dict nombre→función. Por defecto: TOOL_REGISTRY.
        verbose:        Si True, imprime cada paso incluyendo presupuesto restante.
        label:          Etiqueta para los prints.

    Returns:
        Texto de respuesta, o mensaje de presupuesto agotado.
    """
    tools         = tools         if tools         is not None else TOOLS
    tool_registry = tool_registry if tool_registry is not None else TOOL_REGISTRY

    if system is None:
        system = (
            "Eres un asistente útil con acceso a herramientas. "
            "Usa las herramientas SOLO cuando sea estrictamente necesario. "
            "Responde en el idioma del usuario."
        )

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_message},
    ]

    if verbose:
        print(f"\n{'─'*60}")
        print(f"  [{label}] 👤 {user_message[:120]}")
        print(f"  [{label}] 💰 Presupuesto: {max_tool_calls} tool calls")

    # TODO 1: Inicializa un contador de tool calls usadas
    tool_calls_used = ...

    for iteration in range(1, 11):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools if tools else None,
            tool_choice="auto" if tools else None,
        )
        choice        = response.choices[0]
        finish_reason = choice.finish_reason

        if verbose:
            print(f"  [{label}] iter={iteration} finish_reason={finish_reason}")

        if finish_reason == "stop":
            text = choice.message.content or ""
            if verbose:
                preview = text[:200] + "..." if len(text) > 200 else text
                print(f"  [{label}] 🤖 {preview}")
            return text

        if finish_reason == "tool_calls":
            assistant_msg = choice.message
            messages.append(assistant_msg)

            for tc in assistant_msg.tool_calls:

                # TODO 2: Verifica si el presupuesto está agotado ANTES de ejecutar.
                # Si lo está, imprime un aviso y retorna un string descriptivo.
                # Pista: compara tool_calls_used con max_tool_calls
                ...

                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)

                # TODO 3: Incrementa el contador
                ...

                if verbose:
                    # TODO 4: Imprime el presupuesto restante junto al nombre de la tool
                    # Ejemplo: "  [Agente] 🔧 web_search(...) | restante: 2/3"
                    remaining = ...
                    print(f"  [{label}] 🔧 {tool_name}({json.dumps(tool_args, ensure_ascii=False)}) | restante: {remaining}/{max_tool_calls}")

                fn     = tool_registry.get(tool_name)
                result = fn(**tool_args) if fn else f"Error: herramienta '{tool_name}' no encontrada."

                if verbose:
                    preview = result[:150] + "..." if len(result) > 150 else result
                    print(f"  [{label}]    → {preview}")

                messages.append({
                    "role":         "tool",
                    "tool_call_id": tc.id,
                    "content":      result,
                })
        else:
            break

    return "[Agente detenido: límite de iteraciones]"


print("✅ Esqueleto de run_agent_with_budget listo. Completa los TODOs.")

### 🧪 Tests Parte 1

Ejecuta estas celdas para validar tu implementación.
Todas deben imprimir ✅.

In [ ]:
# Test 1.1 — Con presupuesto suficiente, el agente responde normalmente
print("Test 1.1: presupuesto suficiente")
result = run_agent_with_budget(
    "¿Cuánto es 123 * 456?",
    max_tool_calls=3,
    verbose=False,
)
assert isinstance(result, str) and len(result) > 0, "Debe retornar una respuesta no vacía"
assert "56088" in result.replace(",","").replace(".",""), f"La respuesta debe contener el resultado correcto (56088), obtuvo: {result}"
print("✅ Test 1.1 pasado\n")

In [ ]:
# Test 1.2 — Con presupuesto 0, no debe ejecutar ninguna tool call
print("Test 1.2: presupuesto = 0")

tool_calls_log = []
original_web_search = web_search

def web_search_spy(query: str) -> str:
    tool_calls_log.append("web_search")
    return original_web_search(query)

def calculate_spy(expression: str) -> str:
    tool_calls_log.append("calculate")
    return calculate(expression)

spy_registry = {
    "web_search": web_search_spy,
    "get_weather": get_weather,
    "calculate": calculate_spy,
}

tool_calls_log.clear()
result = run_agent_with_budget(
    "¿Cuál es el precio actual de Bitcoin?",
    max_tool_calls=0,
    tool_registry=spy_registry,
    verbose=False,
)
assert len(tool_calls_log) == 0, f"Con max_tool_calls=0 no debe ejecutar ninguna tool, ejecutó: {tool_calls_log}"
print("✅ Test 1.2 pasado\n")

In [ ]:
# Test 1.3 — El agente se detiene exactamente en max_tool_calls
print("Test 1.3: el agente se detiene exactamente en el límite")

tool_calls_log.clear()
result = run_agent_with_budget(
    "Busca el precio de Bitcoin en USD y luego busca el precio de Ethereum en USD.",
    max_tool_calls=1,
    tool_registry=spy_registry,
    verbose=True,
)
assert len(tool_calls_log) <= 1, f"No debe superar max_tool_calls=1, ejecutó {len(tool_calls_log)}: {tool_calls_log}"
print(f"✅ Test 1.3 pasado — tool calls ejecutadas: {len(tool_calls_log)}\n")

In [ ]:
# Test 1.4 — El contador cuenta tool calls individuales, no iteraciones
print("Test 1.4: el contador cuenta tool calls, no iteraciones del loop")

tool_calls_log.clear()
result = run_agent_with_budget(
    "¿Cuánto es 10 * 5? y también ¿cuánto es 20 + 30?",
    max_tool_calls=2,
    tool_registry=spy_registry,
    verbose=True,
)
assert len(tool_calls_log) <= 2, f"No debe superar 2 tool calls, ejecutó: {len(tool_calls_log)}"
print(f"✅ Test 1.4 pasado — tool calls ejecutadas: {len(tool_calls_log)}\n")

---
## 📝 PARTE 2: Controlando el Comportamiento con el System Prompt

### Contexto

El system prompt es la principal palanca para controlar cómo razona el agente.
Pequeños cambios en el wording producen comportamientos muy diferentes.

En esta parte escribirás **tres system prompts** distintos para el mismo agente
y observarás cómo cambia su comportamiento. No tocas el loop — solo el texto.

### Tu tarea

Escribe los system prompts para lograr cada comportamiento descrito.
Luego corre las celdas de validación.

### 2.1 — Agente Conservador

**Comportamiento deseado:** el agente debe responder preguntas de conocimiento
general **sin usar ninguna tool**, incluso si la pregunta menciona términos
técnicos o recientes. Solo debe usar tools si el usuario pregunta explícitamente
por datos en tiempo real (clima, precios en vivo, noticias de hoy).

**Consulta de prueba:** `"¿Qué es LangGraph y para qué sirve?"`
**Resultado esperado:** responde desde su conocimiento, `finish_reason=stop`
en la primera iteración, sin ningún tool call.

In [ ]:
# TODO 5: Escribe el system prompt para el agente conservador
SYSTEM_CONSERVADOR = """
...
"""

In [ ]:
# Validación 2.1
print("Validación 2.1: Agente Conservador")
tool_calls_log.clear()
result = run_agent_with_budget(
    "¿Qué es LangGraph y para qué sirve?",
    max_tool_calls=5,
    system=SYSTEM_CONSERVADOR,
    tool_registry=spy_registry,
    verbose=True,
)
assert len(tool_calls_log) == 0, (
    f"❌ El agente conservador NO debe usar tools para esta pregunta.\n"
    f"   Usó: {tool_calls_log}\n"
    f"   Revisa tu system prompt — debe indicar claramente cuándo NO usar tools."
)
assert len(result) > 50, "La respuesta debe tener contenido sustancial"
print(f"✅ Validación 2.1 pasada — respondió sin tools ({len(result)} chars)\n")

### 2.2 — Agente Exhaustivo

**Comportamiento deseado:** para cualquier pregunta que involucre un tema técnico
o nombre propio, el agente SIEMPRE debe buscar en la web primero, aunque crea
conocer la respuesta. Debe mencionar explícitamente en su respuesta que verificó
la información.

**Consulta de prueba:** `"¿Quién creó Python?"`
**Resultado esperado:** usa `web_search` antes de responder, aunque "Guido van Rossum"
sea conocimiento general.

In [ ]:
# TODO 6: Escribe el system prompt para el agente exhaustivo
SYSTEM_EXHAUSTIVO = """
...
"""

In [ ]:
# Validación 2.2
print("Validación 2.2: Agente Exhaustivo")
tool_calls_log.clear()
result = run_agent_with_budget(
    "¿Quién creó Python?",
    max_tool_calls=5,
    system=SYSTEM_EXHAUSTIVO,
    tool_registry=spy_registry,
    verbose=True,
)
assert "web_search" in tool_calls_log, (
    f"❌ El agente exhaustivo DEBE usar web_search para esta pregunta.\n"
    f"   Tools usadas: {tool_calls_log}\n"
    f"   Revisa tu system prompt — debe instruir buscar siempre."
)
print(f"✅ Validación 2.2 pasada — usó web_search como se esperaba\n")

### 2.3 — Agente Eficiente (el más desafiante)

**Comportamiento deseado:** cuando el usuario hace una pregunta que requiere
búsqueda Y cálculo, el agente debe hacer **exactamente una** búsqueda y
**exactamente un** cálculo — no más. Si ya tiene el dato, no vuelve a buscarlo.

**Consulta de prueba:**
`"Busca el tipo de cambio actual USD a COP y calcula cuántos pesos son 350 dólares"`

**Resultado esperado:** exactamente 2 tool calls: una a `web_search` y una a `calculate`.
No debe buscar dos veces ni calcular dos veces.

In [ ]:
# TODO 7: Escribe el system prompt para el agente eficiente
SYSTEM_EFICIENTE = """
...
"""

In [ ]:
# Validación 2.3
print("Validación 2.3: Agente Eficiente")
tool_calls_log.clear()
result = run_agent_with_budget(
    "Busca el tipo de cambio actual USD a COP y calcula cuántos pesos son 350 dólares.",
    max_tool_calls=5,
    system=SYSTEM_EFICIENTE,
    tool_registry=spy_registry,
    verbose=True,
)
web_calls = tool_calls_log.count("web_search")
calc_calls = tool_calls_log.count("calculate")
assert web_calls == 1, (
    f"❌ Debe hacer exactamente 1 web_search, hizo {web_calls}.\n"
    f"   Tools usadas: {tool_calls_log}"
)
assert calc_calls == 1, (
    f"❌ Debe hacer exactamente 1 calculate, hizo {calc_calls}.\n"
    f"   Tools usadas: {tool_calls_log}"
)
print(f"✅ Validación 2.3 pasada — web_search: {web_calls}x, calculate: {calc_calls}x\n")

---
## 🏁 Resultado Final

Si todos los tests pasaron, ejecuta esta celda para ver tu resumen:

In [ ]:
print("=" * 60)
print("RESUMEN DEL EJERCICIO")
print("=" * 60)
print("""
Parte 1 — Presupuesto de Tool Calls:
  ✅ Implementaste max_tool_calls en el agentic loop
  ✅ El contador opera a nivel de tool calls individuales, no iteraciones
  ✅ El agente se detiene limpiamente cuando agota el presupuesto

Parte 2 — Control por System Prompt:
  ✅ Agente Conservador: no usa tools para conocimiento general
  ✅ Agente Exhaustivo:  siempre verifica con búsqueda
  ✅ Agente Eficiente:   exactamente una tool call por tipo

Reflexión final:
  → El mismo loop puede comportarse de maneras radicalmente distintas
    según cómo instruyas al modelo en el system prompt.
  → El presupuesto de tool calls es una forma de "guardarrail" externo
    que no depende del modelo — funciona independientemente del prompt.
  → En producción, necesitas AMBOS: buen prompt + límites en el código.
""")

---
## 💡 Preguntas de Reflexión (no hay código que entregar)

1. En el Test 1.3 forzaste al agente a detenerse con presupuesto=1.
   ¿La respuesta que retornó fue útil? ¿Qué dice eso sobre el diseño del agente?

2. ¿Qué fue más difícil de lograr: el Agente Conservador o el Eficiente?
   ¿Por qué crees que un tipo de instrucción es más fácil de seguir para el modelo?

3. Imagina que `max_tool_calls` y el system prompt dan instrucciones contradictorias
   (el prompt dice "busca siempre" pero el presupuesto es 0).
   ¿Cuál de los dos debería ganar? ¿Por qué?